In [2]:
import scanpy as sc
import numpy as np
import scipy as sp
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import colors
import seaborn as sb
import logging
import importlib
import warnings
warnings.filterwarnings("ignore")
import pickle as pkl
from matplotlib.colors import LinearSegmentedColormap
import os
os.chdir(r"G:\My Drive\result")

In [3]:
adata_all = sc.read_h5ad("250307_gut_liver_blood_ultimate_annotated.h5ad")

In [5]:
adata_all.obs['donor+tissue+celltype'] = adata_all.obs['Donor ID'].astype(str).map(str) + ' ' + adata_all.obs['tissue'].astype(str).map(str) + ' ' + adata_all.obs['celltype'].astype(str).map(str)

In [7]:
adata_TCR = adata_all[adata_all.obs['chain_pairing'].isin(["single pair", "extra VJ","extra VDJ","two full chains"]),:]
adata_ab = adata_TCR[adata_TCR.obs['general type'].isin(['TCRab CD4','TCRab CD8aa','TCRab CD8ab']),:]
adata_ab.obs['clone_code'] = adata_ab.obs['TRAV'].astype(str).map(str)+' '+adata_ab.obs['TRBV'].astype(str).map(str)+' '+adata_ab.obs['cdr3a'].astype(str).map(str)+' '+ adata_ab.obs['cdr3b'].astype(str).map(str)
### Assign clones
adata3 = adata_ab[adata_ab.obs['batch'] == '3',:]
adata4 = adata_ab[adata_ab.obs['batch'] == '4',:]
adata5 = adata_ab[adata_ab.obs['batch'] == '5',:]
adatas=[adata3,adata4,adata5]
for i in range(3):
    adatas[i].obs['clone'] = adatas[i].obs.groupby('clone_code').ngroup().astype('str')
    adatas[i].obs['clone']='clone '+ adatas[i].obs['clone'].astype(str).map(str)
###  identify top expanded clones in each tissue

for i in [0,1,2]:
    for j in set( adatas[i].obs['tissue']):
        adata_t = adatas[i][adatas[i].obs['tissue']==j,:]
        counts = pd.DataFrame(adata_t.obs['clone'].value_counts())
        #cut_id = int(np.ceil(0.05*len(pd.DataFrame(adata_t.obs['clone'].value_counts()))))
        counts['count'][0:10] = '1-10'
        counts['count'][10:100] = '11-100'
        counts['count'][100:] = '100+'
        
        adatas[i].obs[j+' top clones'] = adatas[i].obs['clone']
        for cloneid in set(adata_t.obs['clone']):
            adatas[i].obs = adatas[i].obs.replace({j+' top clones' : { cloneid: counts.loc[cloneid,'count']}})
        adatas[i].obs[j+' top clones'][adatas[i].obs['tissue']!=j] = ''

for i in [0,1,2]:
    for j in set( adatas[i].obs['tissue']):
        adata_t = adatas[i][adatas[i].obs['tissue']==j,:]
        print(adata_t.obs.groupby([j+' top clones'])['general type'].value_counts())

LP top clones  general type
1-10           TCRab CD4         84
               TCRab CD8ab        9
               TCRab CD8aa        0
100+           TCRab CD4       2459
               TCRab CD8ab      238
               TCRab CD8aa       33
11-100         TCRab CD4        231
               TCRab CD8ab       24
               TCRab CD8aa       18
Name: count, dtype: int64
IEL top clones  general type
1-10            TCRab CD8ab      141
                TCRab CD4         30
100+            TCRab CD8ab     1184
                TCRab CD4        907
11-100          TCRab CD8ab      306
                TCRab CD4         81
Name: count, dtype: int64
L top clones  general type
1-10          TCRab CD8aa      302
              TCRab CD8ab      109
              TCRab CD4         23
100+          TCRab CD4       1213
              TCRab CD8ab     1079
              TCRab CD8aa      466
11-100        TCRab CD8aa      252
              TCRab CD8ab      211
              TCRab CD4         38
Nam

In [35]:
gene2tx = pd.read_csv(r"C:\Users\andre\Documents\GitHub\gut-liver-TRM\Revision\Geneformer_finetunning_hyperopt\features_10x_2020A.tsv",sep = '\t',names = ['Gene stable ID','Gene name','Gene Expression'])
del gene2tx['Gene Expression']
gene2tx.set_index('Gene name',inplace=True)
gene2tx= gene2tx[~gene2tx.index.duplicated(keep='first')]

In [36]:
keynames = [
    'Donor AJG2309 L TCRab CD8ab TRM',
    'Donor AJKQ118 IEL TCRab CD8ab TRM', 
    'Donor AJKQ118 L TCRab CD4 TRM', 
    'Donor AJKQ118 L TCRab CD8ab TRM']

In [38]:
for gtype in ['TCRab CD4','TCRab CD8ab']:
    for j in set(adata_all.obs['tissue']):
        for i in range(0,3):
            adata_temp = adatas[i][adatas[i].obs['general type']==gtype,:]
            adata_t = adata_temp[adata_temp.obs['tissue'] == j, :]
            for ct in set(adata_t.obs['general subtype']):
                keyname = ' '.join([adata_t.obs['Donor ID'][0],j,gtype,ct])
                if keyname in keynames:
                    
                    TRM_cd_t_specific = adata_t[adata_t.obs['donor+tissue+celltype'] == keyname,:]     
                    TRM_cd_t_specific = TRM_cd_t_specific[TRM_cd_t_specific.obs[j+' top clones'].isin(['1-10','100+']),:]
                    TRM_cd_t_specific.obs['top10_or_not']= TRM_cd_t_specific.obs[j+' top clones'] =='1-10'
                    TRM_cd_t_specific.obs['top10_or_not']= TRM_cd_t_specific.obs['top10_or_not'].astype(str)
                    #if keyname == 'Donor AJG2309 L TCRab CD4 TRM':
                        #adata_for_test = TRM_cd_t_specific
                    #sc.pl.dotplot(TRM_cd_t_specific, result_dict, groupby = j+' top clones' ,
                                  #swap_axes = False ,dot_min = 0.1,standard_scale = None, title = keyname, save = keyname+'most_expand_DE.png' )
                    
                    
                    TRM_cd_t_specific = TRM_cd_t_specific[:,TRM_cd_t_specific.var_names.isin(gene2tx.index)]
                    ens_id = gene2tx.loc[TRM_cd_t_specific.var_names]
                    TRM_cd_t_specific.var['ensembl_id'] = ens_id
                    
                    TRM_cd_t_specific.X = TRM_cd_t_specific.layers['counts']
                    TRM_cd_t_specific.X = sp.sparse.csc_matrix(TRM_cd_t_specific.X)
                    TRM_cd_t_specific.obs =TRM_cd_t_specific.obs[['top10_or_not','activation','n_counts']]
                    
                    del TRM_cd_t_specific.uns
                    del TRM_cd_t_specific.obsm
                    del TRM_cd_t_specific.layers
                    del TRM_cd_t_specific.obsp
                    
                    TRM_cd_t_specific.obs = TRM_cd_t_specific.obs.astype(str)
                    TRM_cd_t_specific.var = TRM_cd_t_specific.var.astype(str)
                    TRM_cd_t_specific.obs['n_counts'] = TRM_cd_t_specific.obs['n_counts'].astype(float)
                    savepath = r"G:\My Drive\result\publication\cellreport\revision\geneformer_hyperopt\\"
                    TRM_cd_t_specific.write(savepath+keyname+'.h5ad')